In [ ]:
!pip -q install datasets
import os
import numpy as np
import pandas as pd
from google.colab import drive
from datasets import load_dataset
drive.mount("/content/drive")
P = "/content/drive/MyDrive/"
P += "Project2_Stockout/fresh_clean.parquet"
df = pd.read_parquet(P)
need = {"so_allday", "disc_odd", "so_day", "so_share"}
assert need <= set(df.columns), "flags"
assert len(df) == 4_500_000
assert round(df["so_day"].mean(), 4) == 0.4427
assert round(df["so_share"].mean(), 4) == 0.1988
tr = load_dataset("Dingdong-Inc/FreshRetailNet-50K")["train"]

Mounted at /content/drive


README.md:   0%|          | 0.00/4.88k [00:00<?, ?B/s]

data/train.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/train.parquet: downloading bytes:           |  0.00B            

data/eval.parquet: reconstructing file:   0%|          |  0.00B / 8.44MB            

data/eval.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/4500000 [00:00<?, ? examples/s]

Generating eval split:   0%|          | 0/350000 [00:00<?, ? examples/s]

In [ ]:
a = df[~df["so_allday"]]
print(len(a))
print(round(df["so_day"].mean(), 4),
      round(df["so_share"].mean(), 4))
print(round(a["so_day"].mean(), 4),
      round(a["so_share"].mean(), 4))

4318209
0.4427 0.1988
0.4192 0.165


In [ ]:
day = df.groupby("dt")["so_share"].mean()
day_a = a.groupby("dt")["so_share"].mean()
for s in (day, day_a):
    print(round(s.head(14).mean(), 4),
          round(s.tail(14).mean(), 4))
print(day.idxmax().date(), round(day.max(), 4))
print(day.idxmin().date(), round(day.min(), 4))
dow = df.groupby(df["dt"].dt.dayofweek)["so_share"]
print(dow.mean().round(4))

0.2469 0.188
0.2052 0.1618
2024-04-08 0.3714
2024-05-30 0.1336
dt
0    0.2015
1    0.2081
2    0.1953
3    0.1871
4    0.1755
5    0.1992
6    0.2244
Name: so_share, dtype: float64


In [ ]:
def rank(col, k):
    g = df.groupby(col).agg(
        rows=("so_share", "size"),
        so_share=("so_share", "mean"),
        so_day=("so_day", "mean"))
    g = g[g["rows"] >= 1000]
    return g.sort_values(
        "so_share", ascending=False).head(k)
cat = rank("first_category_id", 10)
print(cat.round(4))
print(rank("city_id", 18).round(4))
c = cat.index[0]
x = df[df["first_category_id"] == c]
print(c, len(x),
      round(x["stock_hour6_22_cnt"].sum()
            / (16 * len(x)), 4))

                     rows  so_share  so_day
first_category_id                          
28                 179820    0.2777  0.4160
29                 314640    0.2578  0.6513
10                 255420    0.2184  0.3882
16                 645930    0.2007  0.5145
4                  979920    0.2002  0.4283
20                 497970    0.1982  0.4365
21                 255870    0.1924  0.4677
15                  62280    0.1855  0.4583
0                   32850    0.1854  0.4947
12                   3240    0.1780  0.5052
            rows  so_share  so_day
city_id                           
3         238320    0.2256  0.4102
9          17730    0.2200  0.5140
14         44370    0.2150  0.4522
13        210240    0.2144  0.4721
1          40590    0.2080  0.4816
6         214560    0.2073  0.4491
8           5850    0.2060  0.3757
12        460530    0.2058  0.4478
2          36630    0.2039  0.5029
7          39960    0.2039  0.4662
15         72270    0.2014  0.4745
16        407520 

In [ ]:
hs = tr.select_columns(["hours_stock_status"])
allday = df["so_allday"].to_numpy()
Ha, Hp, i = np.zeros(24), np.zeros(24), 0
for b in hs.iter(batch_size=50_000):
    m = np.array(b["hours_stock_status"],
                 dtype=np.int8)
    k = allday[i:i + len(m)]
    Ha += m.sum(0)
    Hp += m[~k].sum(0)
    i += len(m)
assert i == len(df)
r_all = Ha[6:22] / len(df)
r_part = Hp[6:22] / (~allday).sum()
assert abs(r_all.mean()
           - df["so_share"].mean()) < 1e-6
assert abs(r_part.mean()
           - a["so_share"].mean()) < 1e-6
print(pd.DataFrame({
    "hour": np.arange(6, 22),
    "all": r_all.round(3),
    "no_allday": r_part.round(3)
}).to_string(index=False))

 hour   all  no_allday
    6 0.062      0.023
    7 0.050      0.010
    8 0.057      0.017
    9 0.071      0.032
   10 0.094      0.056
   11 0.123      0.086
   12 0.148      0.112
   13 0.166      0.131
   14 0.180      0.145
   15 0.199      0.166
   16 0.240      0.208
   17 0.293      0.264
   18 0.336      0.308
   19 0.364      0.337
   20 0.387      0.362
   21 0.410      0.385


In [ ]:
g = ["city_id", "store_id", "product_id"]
m = df.groupby(g)["so_share"].transform("mean")
df["so_dm"] = df["so_share"] - m
for c in ["activity_flag", "holiday_flag"]:
    print(df.groupby(c).agg(
        rows=("so_share", "size"),
        raw=("so_share", "mean"),
        within=("so_dm", "mean")).round(4))
ok = df[~df["disc_odd"]]
d = pd.cut(ok["discount"],
           [0, 0.7, 0.85, 0.95, 1.0])
print(ok.groupby(d, observed=True).agg(
    rows=("so_share", "size"),
    raw=("so_share", "mean"),
    within=("so_dm", "mean")).round(4))
for c in ["precpt", "avg_temperature"]:
    q = pd.qcut(df[c], 4, duplicates="drop")
    print(df.groupby(q, observed=True).agg(
        raw=("so_share", "mean"),
        within=("so_dm", "mean")).round(4))

                  rows     raw  within
activity_flag                         
0              2797107  0.1946 -0.0085
1              1702893  0.2056  0.0140
                 rows     raw  within
holiday_flag                         
0             2950000  0.1946 -0.0042
1             1550000  0.2068  0.0080
                 rows     raw  within
discount                             
(0.0, 0.7]     302905  0.2031  0.0104
(0.7, 0.85]    803079  0.2021  0.0139
(0.85, 0.95]   901641  0.1994  0.0039
(0.95, 1.0]   2476302  0.1961 -0.0071
                    raw  within
precpt                         
(-0.001, 1.429]  0.1947 -0.0050
(1.429, 2.298]   0.1954 -0.0006
(2.298, 4.482]   0.2106  0.0118
(4.482, 42.5]    0.1943 -0.0061
                                raw  within
avg_temperature                            
(12.059000000000001, 19.39]  0.2361  0.0390
(19.39, 22.5]                0.1984  0.0004
(22.5, 25.4]                 0.1688 -0.0296
(25.4, 30.88]                0.1914 -0.0102


In [ ]:
# Before writing insights: all-day robustness, date-level flags, temperature against time, category spread
a2 = df[~df["so_allday"]].copy()
a2["so_dm"] = a2["so_share"] - a2.groupby(g)["so_share"].transform("mean")
print(a2.groupby("activity_flag")["so_dm"].mean().round(4))
print(a2.groupby("holiday_flag")["so_dm"].mean().round(4))
print(df.groupby("dt")["holiday_flag"].nunique().max(), df.groupby("dt")["holiday_flag"].mean().sum())
t14 = df.groupby("dt")["avg_temperature"].mean()
print(round(t14.head(14).mean(), 1), round(t14.tail(14).mean(), 1),
      round(np.corrcoef(t14.to_numpy(), day.to_numpy())[0, 1], 2))
c2 = a2.groupby("first_category_id").agg(rows=("so_share", "size"), so_share=("so_share", "mean"))
print(c2[c2["rows"] >= 1000].sort_values("so_share", ascending=False).head(5).round(4))
allc = df.groupby("first_category_id")["so_share"].agg(["size", "mean"])
allc = allc[allc["size"] >= 1000]
print(round(allc["mean"].min(), 4), round(allc["mean"].max(), 4), len(allc))

activity_flag
0   -0.0088
1    0.0142
Name: so_dm, dtype: float64
holiday_flag
0   -0.0060
1    0.0114
Name: so_dm, dtype: float64
1 31.0
17.5 27.2 -0.51
                     rows  so_share
first_category_id                  
29                 311230    0.2496
16                 643872    0.1982
0                   32687    0.1814
15                  61744    0.1784
27                  20491    0.1682
0.1305 0.2777 28


In [ ]:
# Categories with at least 100 series (9,000 rows), with and without all-day rows
def rank2(d, col, k, minrows=9000):
    g = d.groupby(col).agg(rows=("so_share", "size"), so_share=("so_share", "mean"), so_day=("so_day", "mean"))
    g = g[g["rows"] >= minrows]
    return g.sort_values("so_share", ascending=False).head(k)
both = rank2(df, "first_category_id", 32).join(rank2(a, "first_category_id", 32), lsuffix="_all", rsuffix="_part")
both["allday_pct"] = df.groupby("first_category_id")["so_allday"].mean()
print(both.sort_values("so_share_part", ascending=False).round(4))

                   rows_all  so_share_all  so_day_all  rows_part  \
first_category_id                                                  
29                   314640        0.2578      0.6513     311230   
16                   645930        0.2007      0.5145     643872   
0                     32850        0.1854      0.4947      32687   
15                    62280        0.1855      0.4583      61744   
27                    20700        0.1766      0.5021      20491   
21                   255870        0.1924      0.4677     247841   
18                   211410        0.1701      0.4116     209571   
30                    60300        0.1735      0.4160      59348   
4                    979920        0.2002      0.4283     933314   
31                   172080        0.1766      0.3925     168597   
23                    27090        0.1655      0.4472      26859   
24                   124470        0.1637      0.4037     123479   
8                    259650        0.1777      0

In [ ]:
# Remove each date's overall level, then compare activity and holiday days
def date_ctl(d):
    dm = d.groupby("dt")["so_share"].transform("mean")
    return d["so_dm"] - (dm - d["so_share"].mean())
df["so_dm2"] = date_ctl(df)
a2["so_dm2"] = date_ctl(a2)
print("all rows")
print(df.groupby("activity_flag")["so_dm2"].mean().round(4))
print(df.groupby("holiday_flag")["so_dm2"].mean().round(4))
print("no all-day rows")
print(a2.groupby("activity_flag")["so_dm2"].mean().round(4))
print(a2.groupby("holiday_flag")["so_dm2"].mean().round(4))

all rows
activity_flag
0   -0.0090
1    0.0148
Name: so_dm2, dtype: float64
holiday_flag
0   -0.0
1   -0.0
Name: so_dm2, dtype: float64
no all-day rows
activity_flag
0   -0.0089
1    0.0144
Name: so_dm2, dtype: float64
holiday_flag
0    0.0001
1   -0.0001
Name: so_dm2, dtype: float64


In [ ]:
# Insight 1: time of day
h = np.arange(6, 22)
la, ha_ = r_all.argmin(), r_all.argmax()
lp, hp = r_part.argmin(), r_part.argmax()
print(f"Insight 1: Without all-day rows the stock-out share rises from {r_part[lp]:.1%} at {h[lp]}:00 to "
      f"{r_part[hp]:.1%} at {h[hp]}:00 (with them: {r_all[la]:.1%} to {r_all[ha_]:.1%}). The data cannot show "
      f"whether this is demand building or stock not being replenished, so check evening stock levels first.")

# Insight 2: the category is picked by the data, with a guard against calling a below-average one "high"
best = both["so_share_part"].idxmax()
b = both.loc[best]
assert b["so_share_all"] > df["so_share"].mean(), "category is not above average"
print(f"Insight 2: Category {best} has a high stock-out share with and without all-day rows: {b['so_share_all']:.1%} "
      f"and {b['so_share_part']:.1%} (overall {df['so_share'].mean():.1%} and {a['so_share'].mean():.1%}), on "
      f"{b['rows_all']:,.0f} store-product-days ({b['rows_all']/90:,.0f} series). Only {b['allday_pct']:.1%} of its days "
      f"are all-day stock-outs. Review it first. Categories are encoded IDs, so names are unknown.")

# Insight 3: activity days, after removing each product's own average and each date's level
w, wa = df.groupby("activity_flag")["so_dm2"].mean(), a2.groupby("activity_flag")["so_dm2"].mean()
gap, gap_a = (w[1] - w[0]) * 100, (wa[1] - wa[0]) * 100
kind = "still" if gap_a >= 1 else "only"
print(f"Insight 3: After removing each product's own average and each date's overall level, activity days {kind} show a "
      f"stock-out share {gap_a:.1f} points higher without all-day rows ({gap:.1f} with them), about "
      f"{gap_a / 100 / a['so_share'].mean():.0%} of the {a['so_share'].mean():.1%} baseline. This is an association: "
      f"without delivery records, promotions may simply coincide with demand surges.")

# Trend and day of week
d0, d1 = day.head(14).mean(), day.tail(14).mean()
e0, e1 = day_a.head(14).mean(), day_a.tail(14).mean()
dmean = dow.mean()
names = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
print(f"Trend: The stock-out share falls from {d0:.1%} in the first 14 days to {d1:.1%} in the last 14, and from "
      f"{e0:.1%} to {e1:.1%} without all-day rows. The peak day is {day.idxmax().date()} at {day.max():.1%}. "
      f"{names[dmean.idxmax()]} is highest ({dmean.max():.1%}) and {names[dmean.idxmin()]} lowest ({dmean.min():.1%}). "
      f"This coincides with warming over the same weeks, so the two cannot be separated.")

# Cities
cy = df.groupby("city_id")["so_share"].agg(["size", "mean"])
cy = cy[cy["size"] >= 1000]
big = df["city_id"].value_counts(normalize=True)
print(f"Cities: the {len(cy)} cities range from {cy['mean'].min():.1%} to {cy['mean'].max():.1%} "
      f"({(cy['mean'].max() - cy['mean'].min()) * 100:.1f} points). City {big.index[0]} holds {big.iloc[0]:.1%} of all rows.")

Insight 1: Without all-day rows the stock-out share rises from 1.0% at 7:00 to 38.5% at 21:00 (with them: 5.0% to 41.0%). The data cannot show whether this is demand building or stock not being replenished, so check evening stock levels first.
Insight 2: Category 29 has a high stock-out share with and without all-day rows: 25.8% and 25.0% (overall 19.9% and 16.5%), on 314,640 store-product-days (3,496 series). Only 1.1% of its days are all-day stock-outs. Review it first. Categories are encoded IDs, so names are unknown.
Insight 3: After removing each product's own average and each date's overall level, activity days still show a stock-out share 2.3 points higher without all-day rows (2.4 with them), about 14% of the 16.5% baseline. This is an association: without delivery records, promotions may simply coincide with demand surges.
Trend: The stock-out share falls from 24.7% in the first 14 days to 18.8% in the last 14, and from 20.5% to 16.2% without all-day rows. The peak day is 2024

In [ ]:
# The category that ranks first with all rows, and how much of that comes from all-day stock-outs
top_all = both["so_share_all"].idxmax()
t = both.loc[top_all]
print(f"Category {top_all} ranks first with all rows ({t['so_share_all']:.1%}) but {t['so_share_part']:.1%} "
      f"without them; {t['allday_pct']:.1%} of its days are all-day stock-outs, which supply "
      f"{t['allday_pct'] / t['so_share_all']:.0%} of its out-of-stock hours. "
      f"Find out whether those days mean the product was not offered.")

Category 28 ranks first with all rows (27.8%) but 13.0% without them; 17.0% of its days are all-day stock-outs, which supply 61% of its out-of-stock hours. Find out whether those days mean the product was not offered.


In [ ]:
print(f"Temperature rises from {t14.head(14).mean():.1f}°C in the first 14 days to {t14.tail(14).mean():.1f}°C in the last 14 "
      f"(correlation with the daily stock-out share {np.corrcoef(t14.to_numpy(), day.to_numpy())[0, 1]:.2f}), "
      f"so temperature cannot be separated from the date trend.")

Temperature rises from 17.5°C in the first 14 days to 27.2°C in the last 14 (correlation with the daily stock-out share -0.51), so temperature cannot be separated from the date trend.


In [ ]:
D = "/content/drive/MyDrive/"
D += "Project2_Stockout/dash/"
os.makedirs(D, exist_ok=True)
a = df[~df["so_allday"]]
hour = pd.DataFrame({
    "hour": np.arange(6, 22),
    "share_all": r_all,
    "share_no_allday": r_part})
assert len(hour) == 16
assert abs(hour["share_all"].mean()
           - df["so_share"].mean()) < 1e-6
hour.to_csv(D + "hourly.csv", index=False)

In [ ]:
def catg(d):
    return d.groupby("first_category_id").agg(
        rows=("so_share", "size"),
        so_share=("so_share", "mean"),
        so_day=("so_day", "mean"))
c = catg(df).join(
    catg(a), lsuffix="_all", rsuffix="_part")
c["series"] = c["rows_all"] / 90
c["allday_pct"] = df.groupby(
    "first_category_id")["so_allday"].mean()
c = c[c["series"] >= 100].reset_index()
c["category"] = ("Cat "
    + c["first_category_id"].astype(str))
c = c.sort_values(
    "so_share_part", ascending=False)
assert len(c) == 22
print(c.head(3).round(4))
c.to_csv(D + "categories.csv", index=False)

    first_category_id  rows_all  so_share_all  so_day_all  rows_part  \
19                 29    314640        0.2578      0.6513     311230   
9                  16    645930        0.2007      0.5145     643872   
0                   0     32850        0.1854      0.4947      32687   

    so_share_part  so_day_part  series  allday_pct category  
19         0.2496       0.6475  3496.0      0.0108   Cat 29  
9          0.1982       0.5129  7177.0      0.0032   Cat 16  
0          0.1814       0.4922   365.0      0.0050    Cat 0  


In [ ]:
d1 = df.groupby("dt")["so_share"].mean()
d2 = a.groupby("dt")["so_share"].mean()
t1 = df.groupby("dt")["avg_temperature"].mean()
daily = pd.DataFrame({
    "dt": d1.index.strftime("%Y%m%d"),
    "share_all": d1.values,
    "share_no_allday": d2.reindex(
        d1.index).values,
    "temp_c": t1.values})
assert len(daily) == 90
print(daily.loc[daily["share_all"].idxmax(), "dt"])
daily.to_csv(D + "daily.csv", index=False)

20240408


In [ ]:
def gap(d):
    k = ["city_id", "store_id", "product_id"]
    m = d.groupby(k)["so_share"].transform("mean")
    t = d.groupby("dt")["so_share"].transform("mean")
    x = d["so_share"] - m - t + d["so_share"].mean()
    g = x.groupby(d["activity_flag"]).mean()
    return g[1] - g[0]
ids = df[["city_id", "store_id", "product_id"]]
kpi = pd.DataFrame([{
    "hours_share_all": df["so_share"].mean(),
    "hours_share_no_allday": a["so_share"].mean(),
    "day_rate_all": df["so_day"].mean(),
    "day_rate_no_allday": a["so_day"].mean(),
    "activity_gap_all": gap(df),
    "activity_gap_no_allday": gap(a),
    "rows": len(df),
    "series": len(ids.drop_duplicates())}])
print(kpi.T.round(4))
assert abs(kpi.at[0, "activity_gap_no_allday"]
           - 0.0233) < 0.001
kpi.to_csv(D + "kpi.csv", index=False)
print(os.listdir(D))

                                   0
hours_share_all         1.988000e-01
hours_share_no_allday   1.650000e-01
day_rate_all            4.427000e-01
day_rate_no_allday      4.192000e-01
activity_gap_all        2.380000e-02
activity_gap_no_allday  2.330000e-02
rows                    4.500000e+06
series                  5.000000e+04
['hourly.csv', 'categories.csv', 'daily.csv', 'kpi.csv']


In [ ]:
h7 = hour.loc[hour["hour"] == 7,
              "share_no_allday"].iloc[0]
h21 = hour.loc[hour["hour"] == 21,
               "share_no_allday"].iloc[0]
top = c.iloc[0]
base = a["so_share"].mean()
headline = (
    f"Stock-outs build through the day, from "
    f"{h7:.1%} of store-product-days at 7:00 to "
    f"{h21:.1%} at 21:00, and Category "
    f"{int(top['first_category_id'])} runs out "
    f"most: {top['so_share_part']:.1%} of "
    f"daytime hours against {base:.1%} overall.")
print(headline)

Stock-outs build through the day, from 1.0% of store-product-days at 7:00 to 38.5% at 21:00, and Category 29 runs out most: 25.0% of daytime hours against 16.5% overall.


In [ ]:
# Footer text for the dashboard; copy the printed lines
w = df["stock_hour6_22_cnt"]
share_hours = w[df["so_allday"]].sum() / w.sum()
print(f"Limits: {len(df):,} store-product-days, {len(ids.drop_duplicates()):,} store-product series, "
      f"one retailer in China, {df['dt'].nunique()} days "
      f"({df['dt'].min():%d %b %Y} to {df['dt'].max():%d %b %Y}); it does not describe Kenya. "
      f"Categories and stores are encoded IDs, so names are unknown. "
      f"Findings are associations, not causes: there are no delivery or replenishment records. "
      f"All-day stock-outs ({df['so_allday'].mean():.1%} of rows) hold {share_hours:.1%} of out-of-stock hours "
      f"and may mean the product was not offered, so rates are shown with and without them.")
print("Source: FreshRetailNet-50K (Dingdong-Inc), Hugging Face, CC BY 4.0; paper arXiv 2505.16319.")

Limits: 4,500,000 store-product-days, 50,000 store-product series, one retailer in China, 90 days (28 Mar 2024 to 25 Jun 2024); it does not describe Kenya. Categories and stores are encoded IDs, so names are unknown. Findings are associations, not causes: there are no delivery or replenishment records. All-day stock-outs (4.0% of rows) hold 20.3% of out-of-stock hours and may mean the product was not offered, so rates are shown with and without them.
Source: FreshRetailNet-50K (Dingdong-Inc), Hugging Face, CC BY 4.0; paper arXiv 2505.16319.


In [ ]:
base = a["so_share"].mean()
h7 = hour.loc[hour["hour"] == 7,
              "share_no_allday"].iloc[0]
h21 = hour.loc[hour["hour"] == 21,
               "share_no_allday"].iloc[0]
top = c.iloc[0]
gap_a = kpi.at[0, "activity_gap_no_allday"]
ns = int(kpi.at[0, "series"])
d_a = a.groupby("dt")["so_share"].mean()
t0, t1 = d_a.head(14).mean(), d_a.tail(14).mean()
tp = df.groupby("dt")["avg_temperature"].mean()
cy = df.groupby("city_id")["so_share"].mean()
w = df["stock_hour6_22_cnt"]
ad_sh = w[df["so_allday"]].sum() / w.sum()

In [ ]:
problem = (
    f"Which product categories and hours of "
    f"the day have the highest stock-out "
    f"rates, and what coincides with them? "
    f"{len(df):,} store-product-days, "
    f"{ns:,} series, "
    f"{df['dt'].min():%d %b %Y} to "
    f"{df['dt'].max():%d %b %Y}.")
method = (
    f"I cleaned and checked the data in pandas, "
    f"measured the share of the 16 daytime "
    f"hours (6:00 to 21:00) each product was "
    f"out of stock, and compared hours, "
    f"categories and days, with and without "
    f"the {df['so_allday'].mean():.1%} of rows "
    f"that are out of stock all day.")
insight = (
    f"Without all-day rows, stock-outs build through the day, from "
    f"{h7:.1%} of store-product-days out of stock at 7:00 to {h21:.1%} at 21:00. "
    f"Among the {len(c)} categories with 100+ series, "
    f"Category {int(top['first_category_id'])} runs out most: "
    f"{top['so_share_part']:.1%} of daytime hours against {base:.1%} overall, "
    f"and {top['so_day_part']:.0%} of its store-product-days see a stock-out. "
    f"Activity days run {gap_a*100:.1f} points higher for the same product "
    f"on the same date.")
trend = (
    f"Without all-day rows, the share falls from {t0:.1%} in the "
    f"first 14 days to {t1:.1%} in the last 14, while temperature rises from "
    f"{tp.head(14).mean():.1f} to {tp.tail(14).mean():.1f} C, so the two cannot be separated.")
cities = (
    f"The {len(cy)} cities differ little "
    f"({cy.min():.1%} to {cy.max():.1%}); "
    f"city 0 holds "
    f"{(df['city_id'] == 0).mean():.1%} of rows.")
limit = (
    f"Limit: one retailer in China, "
    f"{df['dt'].nunique()} days, encoded IDs, "
    f"no delivery records, so these are "
    f"associations. All-day stock-outs hold "
    f"{ad_sh:.1%} of out-of-stock hours and "
    f"may mean the product was not offered.")
check = (
    f"Without all-day rows the hours share is "
    f"{base:.1%} instead of "
    f"{df['so_share'].mean():.1%}, and the day "
    f"rate {a['so_day'].mean():.1%} instead of "
    f"{df['so_day'].mean():.1%}.")

In [ ]:
rec = (f"Check Category {int(top['first_category_id'])} first: it runs out of {top['so_share_part']:.1%} of daytime hours "
       f"against {base:.1%} overall, so ask its store managers when stock runs low and how often it is replenished. "
       f"Ask the retailer for delivery and replenishment records, since this data cannot separate demand surges from supply gaps.")

In [ ]:
print("\n\n".join([problem, method, insight,
    trend, cities, rec, limit]))
print(check)

Which product categories and hours of the day have the highest stock-out rates, and what coincides with them? 4,500,000 store-product-days, 50,000 series, 28 Mar 2024 to 25 Jun 2024.

I cleaned and checked the data in pandas, measured the share of the 16 daytime hours (6:00 to 21:00) each product was out of stock, and compared hours, categories and days, with and without the 4.0% of rows that are out of stock all day.

Without all-day rows, stock-outs build through the day, from 1.0% of store-product-days out of stock at 7:00 to 38.5% at 21:00. Among the 22 categories with 100+ series, Category 29 runs out most: 25.0% of daytime hours against 16.5% overall, and 65% of its store-product-days see a stock-out. Activity days run 2.3 points higher for the same product on the same date.

Without all-day rows, the share falls from 20.5% in the first 14 days to 16.2% in the last 14, while temperature rises from 17.5 to 27.2 C, so the two cannot be separated.

The 18 cities differ little (18.9%